# 🚀 Tavi — Gemma 3 1B On-Device Fine-Tuning & LiteRT Export

**Tavi** is a 100% on-device personal AI assistant for Android powered by Google Gemma 3 1B IT.
This notebook fine-tunes **Gemma 3 1B IT** using **QLoRA (4-bit)** on Tavi's specialized dataset:
- **51 Native Android Tools** (Calendar, SMS, Dialer, Alarms, Device Settings, Memory)
- **Multi-Step Workflow Planning & Execution**
- **On-Device SQLite Memory Extraction & Recall**
- **Adversarial Prompt Injection Defense & Privacy Boundaries**

After training, the notebook merges the LoRA adapter and exports a dynamic **Int4 LiteRT FlatBuffer (`.litertlm`)** ready to run on any Android phone (including 3-4 GB RAM budget phones like Samsung Galaxy A14).

### 1. Check GPU & Hardware Environment
A free Google Colab **T4 GPU** is sufficient for 4-bit QLoRA training (~4 GB VRAM used).

In [ ]:
!nvidia-smi

### 2. Install Required ML & AI Edge Libraries

In [ ]:
!pip install -q -U torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121
!pip install -q -U transformers datasets peft trl bitsandbytes accelerate
!pip install -q ai-edge-torch

### 3. Authenticate with Hugging Face Hub
Gemma 3 weights are gated on Hugging Face. Log in with your free HF token (ensure you have accepted the Gemma license on https://huggingface.co/google/gemma-3-1b-it).

In [ ]:
from huggingface_hub import login
from google.colab import userdata

try:
    hf_token = userdata.get('HF_TOKEN')
    login(token=hf_token)
except Exception:
    print('Please enter your Hugging Face Access Token:')
    login()

### 4. Clone Tavi Repository or Generate Dataset

In [ ]:
import os
if not os.path.exists('Tavi'):
    !git clone https://github.com/MosetiReagan/Tavi.git
    %cd Tavi
else:
    %cd Tavi
    !git pull

!python3 training/dataset_generator.py --samples 6000 --output_dir training/data

### 5. Inspect Dataset Samples

In [ ]:
import json

with open('training/data/tavi_train.jsonl', 'r') as f:
    sample = json.loads(f.readline())

print(f"Category:    {sample.get('category')}")
print(f"Instruction: {sample.get('instruction')}")
print('Formatted Gemma Turn:')
print(sample.get('text'))

### 6. Run QLoRA Fine-Tuning (SFTTrainer)

In [ ]:
!python3 training/train_lora.py \
    --model_id google/gemma-3-1b-it \
    --train_file training/data/tavi_train.jsonl \
    --val_file training/data/tavi_val.jsonl \
    --output_dir training/output/gemma-3-1b-tavi-lora \
    --epochs 3 \
    --batch_size 4 \
    --grad_accum 4 \
    --lr 2e-4 \
    --lora_r 16 \
    --lora_alpha 32 \
    --max_seq_length 1024

### 7. Test Inference on Trained LoRA Model

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel

base_id = "google/gemma-3-1b-it"
adapter_path = "training/output/gemma-3-1b-tavi-lora"

tokenizer = AutoTokenizer.from_pretrained(base_id)
base_model = AutoModelForCausalLM.from_pretrained(base_id, torch_dtype=torch.float16, device_map="auto")
model = PeftModel.from_pretrained(base_model, adapter_path)

prompt = "<start_of_turn>user\nTurn on the flashlight and schedule lunch with Dennis tomorrow at 1pm<end_of_turn>\n<start_of_turn>model\n"
inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

with torch.no_grad():
    outputs = model.generate(**inputs, max_new_tokens=150, temperature=0.1, do_sample=False)

print(tokenizer.decode(outputs[0], skip_special_tokens=False))

### 8. Merge LoRA Adapters & Export to LiteRT FlatBuffer (`.litertlm`)

In [ ]:
!python3 training/export_litertlm.py \
    --base_model_id google/gemma-3-1b-it \
    --adapter_dir training/output/gemma-3-1b-tavi-lora \
    --merged_dir training/output/gemma-3-1b-tavi-merged \
    --output_file app/src/main/assets/models/gemma-3-1b-it-tavi.litertlm \
    --quantization int4

### 9. Download Compiled Model for Android

In [ ]:
from google.colab import files
import os

target_model = "app/src/main/assets/models/gemma-3-1b-it-tavi.litertlm"
if os.path.exists(target_model):
    print(f"Downloading {target_model} (~650 MB)...")
    files.download(target_model)
else:
    print("File not found. Check export output logs.")